# 02. Exploratory Data Analysis (EDA)

Comprehensive analysis of customer transaction patterns and behaviors.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Load data
df = pd.read_csv('../data/raw/ecommerce_transactions.csv')
df['transaction_date'] = pd.to_datetime(df['transaction_date'])
print(f"Loaded {len(df):,} transactions")

## 1. Data Quality Check

In [ ]:
print("=" * 50)
print("DATA QUALITY REPORT")
print("=" * 50)

print("\n📊 Missing Values:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values ✅")

print(f"\n📊 Duplicates: {df.duplicated().sum()} duplicate rows")
print(f"\n📊 Data Types:")
print(df.dtypes)

In [ ]:
df.describe().round(2)

## 2. Transaction Trends Over Time

In [ ]:
# Monthly trends
df['year_month'] = df['transaction_date'].dt.to_period('M')
monthly_stats = df.groupby('year_month').agg({
    'transaction_id': 'count',
    'total_amount': 'sum',
    'customer_id': 'nunique'
}).rename(columns={
    'transaction_id': 'transactions',
    'total_amount': 'revenue',
    'customer_id': 'unique_customers'
})

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(monthly_stats.index.astype(str), monthly_stats['transactions'], marker='o', linewidth=2, markersize=4)
axes[0].set_ylabel('Transactions', fontsize=11)
axes[0].set_title('Monthly Transaction Volume', fontsize=12, fontweight='bold')
axes[0].fill_between(range(len(monthly_stats)), monthly_stats['transactions'], alpha=0.3)

axes[1].plot(monthly_stats.index.astype(str), monthly_stats['revenue'], marker='o', linewidth=2, markersize=4, color='green')
axes[1].set_ylabel('Revenue ($)', fontsize=11)
axes[1].set_title('Monthly Revenue', fontsize=12, fontweight='bold')
axes[1].fill_between(range(len(monthly_stats)), monthly_stats['revenue'], alpha=0.3, color='green')

axes[2].plot(monthly_stats.index.astype(str), monthly_stats['unique_customers'], marker='o', linewidth=2, markersize=4, color='purple')
axes[2].set_ylabel('Unique Customers', fontsize=11)
axes[2].set_title('Monthly Active Customers', fontsize=12, fontweight='bold')
axes[2].fill_between(range(len(monthly_stats)), monthly_stats['unique_customers'], alpha=0.3, color='purple')

plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../reports/figures/monthly_trends.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Customer Behavior Analysis

In [ ]:
# Per-customer statistics
customer_stats = df.groupby('customer_id').agg({
    'transaction_id': 'count',
    'total_amount': ['sum', 'mean'],
    'transaction_date': ['min', 'max']
})
customer_stats.columns = ['frequency', 'total_spend', 'avg_order_value', 'first_purchase', 'last_purchase']
customer_stats['customer_tenure_days'] = (customer_stats['last_purchase'] - customer_stats['first_purchase']).dt.days

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Purchase frequency distribution
axes[0, 0].hist(customer_stats['frequency'], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[0, 0].set_xlabel('Number of Purchases')
axes[0, 0].set_ylabel('Number of Customers')
axes[0, 0].set_title('Purchase Frequency Distribution', fontweight='bold')
axes[0, 0].axvline(customer_stats['frequency'].median(), color='red', linestyle='--', label=f"Median: {customer_stats['frequency'].median():.0f}")
axes[0, 0].legend()

# Total spend distribution
axes[0, 1].hist(customer_stats['total_spend'], bins=50, color='green', edgecolor='white', alpha=0.8)
axes[0, 1].set_xlabel('Total Spend ($)')
axes[0, 1].set_ylabel('Number of Customers')
axes[0, 1].set_title('Customer Lifetime Value Distribution', fontweight='bold')
axes[0, 1].axvline(customer_stats['total_spend'].median(), color='red', linestyle='--', label=f"Median: ${customer_stats['total_spend'].median():,.0f}")
axes[0, 1].legend()

# AOV distribution
axes[1, 0].hist(customer_stats['avg_order_value'], bins=40, color='orange', edgecolor='white', alpha=0.8)
axes[1, 0].set_xlabel('Average Order Value ($)')
axes[1, 0].set_ylabel('Number of Customers')
axes[1, 0].set_title('Average Order Value Distribution', fontweight='bold')

# Frequency vs Monetary scatter
scatter = axes[1, 1].scatter(customer_stats['frequency'], customer_stats['total_spend'], 
                              c=customer_stats['avg_order_value'], cmap='viridis', alpha=0.5, s=20)
axes[1, 1].set_xlabel('Purchase Frequency')
axes[1, 1].set_ylabel('Total Spend ($)')
axes[1, 1].set_title('Frequency vs Monetary (colored by AOV)', fontweight='bold')
plt.colorbar(scatter, ax=axes[1, 1], label='Avg Order Value')

plt.tight_layout()
plt.savefig('../reports/figures/customer_behavior.png', dpi=150)
plt.show()

## 4. Product Category Analysis

In [ ]:
category_stats = df.groupby('product_category').agg({
    'transaction_id': 'count',
    'total_amount': 'sum',
    'customer_id': 'nunique'
}).rename(columns={
    'transaction_id': 'transactions',
    'total_amount': 'revenue',
    'customer_id': 'unique_customers'
}).sort_values('revenue', ascending=False)

category_stats['avg_transaction'] = category_stats['revenue'] / category_stats['transactions']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Revenue by category
colors = sns.color_palette('husl', len(category_stats))
bars = axes[0].barh(category_stats.index, category_stats['revenue'], color=colors)
axes[0].set_xlabel('Revenue ($)')
axes[0].set_title('Revenue by Product Category', fontweight='bold')
for bar, val in zip(bars, category_stats['revenue']):
    axes[0].text(val + 5000, bar.get_y() + bar.get_height()/2, f'${val/1000:.0f}K', va='center')

# Average transaction by category
axes[1].barh(category_stats.index, category_stats['avg_transaction'], color=colors)
axes[1].set_xlabel('Average Transaction Value ($)')
axes[1].set_title('Avg Transaction by Category', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/category_analysis.png', dpi=150)
plt.show()

category_stats.round(2)

## 5. Geographic Distribution

In [ ]:
country_stats = df.groupby('country').agg({
    'transaction_id': 'count',
    'total_amount': 'sum',
    'customer_id': 'nunique'
}).sort_values('total_amount', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('Blues_r', len(country_stats))
ax.pie(country_stats['total_amount'], labels=country_stats.index, autopct='%1.1f%%', colors=colors, startangle=90)
ax.set_title('Revenue Distribution by Country', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/geographic_dist.png', dpi=150)
plt.show()

## 6. Payment Method Analysis

In [ ]:
payment_stats = df.groupby('payment_method').agg({
    'transaction_id': 'count',
    'total_amount': ['sum', 'mean']
}).round(2)
payment_stats.columns = ['transactions', 'total_revenue', 'avg_transaction']
payment_stats = payment_stats.sort_values('transactions', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(payment_stats.index, payment_stats['transactions'], color=sns.color_palette('husl', len(payment_stats)))
ax.set_xlabel('Payment Method')
ax.set_ylabel('Number of Transactions')
ax.set_title('Transactions by Payment Method', fontweight='bold')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../reports/figures/payment_methods.png', dpi=150)
plt.show()

payment_stats

## 7. Correlation Analysis

In [ ]:
# Numeric columns correlation
numeric_cols = ['quantity', 'unit_price', 'total_amount']
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f', ax=ax)
ax.set_title('Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/correlation_matrix.png', dpi=150)
plt.show()

## Key Insights Summary

In [ ]:
print("=" * 60)
print("KEY INSIGHTS")
print("=" * 60)
print(f"\n📊 Dataset Summary:")
print(f"   • Total transactions: {len(df):,}")
print(f"   • Unique customers: {df['customer_id'].nunique():,}")
print(f"   • Total revenue: ${df['total_amount'].sum():,.2f}")
print(f"   • Avg order value: ${df['total_amount'].mean():.2f}")

print(f"\n📈 Customer Behavior:")
print(f"   • Avg purchases per customer: {customer_stats['frequency'].mean():.1f}")
print(f"   • Median customer lifetime value: ${customer_stats['total_spend'].median():,.2f}")

top_cat = category_stats.index[0]
print(f"\n🛒 Product Categories:")
print(f"   • Top category by revenue: {top_cat}")

top_country = country_stats.index[0]
print(f"\n🌍 Geographic:")
print(f"   • Top market: {top_country}")

print("\n✅ EDA Complete!")